In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

file_path = "/content/drive/MyDrive/archive NLP"

if os.path.exists(file_path):
    print("File exists!")
else:
    print("File NOT found!")


File exists!


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import os

data_directory = "/content/drive/MyDrive/archive NLP"
file_to_load = "train.ft.txt"
full_data_path = os.path.join(data_directory, file_to_load)

texts = []
labels = []

try:
    with open(full_data_path, 'r', encoding='utf-8') as f:
        for line in f:

            if line.startswith('__label__'):
                parts = line.strip().split(' ', 1)
                label_str = parts[0].replace('__label__', '')
                label = int(label_str) - 1
                text = parts[1]
                texts.append(text)
                labels.append(label)
    print(f"Successfully loaded {len(texts)} entries from '{file_to_load}'.")

    data = pd.DataFrame({'review_text': texts, 'sentiment': labels})

    # 2. Train/validation split
    X_train, X_val, y_train, y_val = train_test_split(
        data["review_text"].astype(str).values,
        data["sentiment"].values,
        test_size=0.2,
        random_state=42,
        stratify=data["sentiment"].values
    )

    # 3. Tokenization
    max_words = 10000
    max_len = 100

    tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train)

    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_val_seq = tokenizer.texts_to_sequences(X_val)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")

    print("Data loading, splitting, and tokenization complete.")
    print(f"X_train_pad shape: {X_train_pad.shape}")
    print(f"y_train shape: {y_train.shape}")

except FileNotFoundError:
    print(f"Error: The file '{full_data_path}' was not found. Please ensure the path and filename are correct.")
except Exception as e:
    print(f"An error occurred during data loading or processing: {e}")

Successfully loaded 3600000 entries from 'train.ft.txt'.
Data loading, splitting, and tokenization complete.
X_train_pad shape: (2880000, 100)
y_train shape: (2880000,)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

embedding_dim = 64

model = Sequential([
    Embedding(max_words, embedding_dim, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

model.summary()

history = model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val_pad, y_val),
    verbose=1
)

# Evaluate
loss, acc = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"Validation accuracy: {acc:.3f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
45000/45000 ━━━━━━━━━━━━━━━━━━━━ 399s 9ms/step - accuracy: 0.8068 - loss: 0.3719 - val_accuracy: 0.9398 - val_loss: 0.1590
Epoch 2/5
45000/45000 ━━━━━━━━━━━━━━━━━━━━ 389s 9ms/step - accuracy: 0.9436 - loss: 0.1506 - val_accuracy: 0.9459 - val_loss: 0.1488
Epoch 3/5
45000/45000 ━━━━━━━━━━━━━━━━━━━━ 385s 9ms/step - accuracy: 0.9503 - loss: 0.1346 - val_accuracy: 0.9477 - val_loss: 0.1447
Epoch 4/5
45000/45000 ━━━━━━━━━━━━━━━━━━━━ 385s 9ms/step - accuracy: 0.9549 - loss: 0.1241 - val_accuracy: 0.9478 - val_loss: 0.1467
Epoch 5/5
45000/45000 ━━━━━━━━━━━━━━━━━━━━ 387s 9ms/step - accuracy: 0.9584 - loss: 0.1159 - val_accuracy: 0.9475 - val_loss: 0.1440
Validation accuracy: 0.948


In [ ]:
# Save model
model.save("sentiment_lstm_model.h5")

# Save tokenizer
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved!")


Model and tokenizer saved!


next time

In [ ]:
from tensorflow.keras.models import load_model
import pickle

# Load model
model = load_model("sentiment_lstm_model.h5")

# Load tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

print("Model and tokenizer loaded!")

Model and tokenizer loaded!


In [ ]:
def predict_review_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")
    prob = model.predict(pad)[0][0]
    label = "Positive" if prob >= 0.5 else "Negative"
    return prob, label

example = "The staff were very helpful and the service was excellent."
score, label = predict_review_sentiment(example)
print("Score:", score, "Label:", label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
Score: 0.9775929 Label: Positive
